In [1]:
from dotenv import load_dotenv
load_dotenv()
import os
openai_base_url = os.getenv("OPENAI_API_BASE")
openai_token = os.getenv("OPENAI_API_TOKEN")

In [2]:


from openai import OpenAI
model = OpenAI(api_key = openai_token, base_url = openai_base_url)

### call llm model for testing

client = model.chat.completions.create(messages = [{
    "role" : "user", 
    "content" : [{
        "type" : "text",
        "text" : "what is the capital of india?"
    }]
}
], model="gpt-4o", temperature=0.0)
print(client.choices[0].message.content)

The capital of India is New Delhi.


In [ ]:
from langchain_community.embeddings import OpenAIEmbeddings
model = OpenAIEmbeddings(model = "text-embedding-3-small", api_key = openai_token, base_url = openai_base_url, dimensions = 1024)
import numpy as np
response = np.array(model.embed_query("what is the capital of india?"))
response.shape

/Users/biman_giri/.pyenv/versions/3.11.4/envs/BimanDS/lib/python3.11/site-packages/langchain_community/embeddings/openai.py:271: UserWarning: WARNING! dimensions is not default parameter.
                    dimensions was transferred to model_kwargs.
                    Please confirm that dimensions is what you intended.
  warnings.warn(


(1024,)

In [4]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader("speech.txt")
data = loader.load()
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
chunks = text_splitter.split_documents(data)
chunks_with_content = [chunk.page_content for chunk in chunks]

/Users/biman_giri/.pyenv/versions/3.11.4/envs/BimanDS/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
# encode the chunks
response = model.embed_documents(chunks_with_content)
embeddings = np.array([np.array(r) for r in response])
embeddings.shape
# from sklearn.metrics.pairwise import cosine_similarity
# cosine_similarity(embeddings[0].reshape(1, -1), embeddings[1].reshape(1, -1))


(5, 1024)

In [17]:
### vector store
from langchain_community.vectorstores import Chroma
db = Chroma.from_documents(documents = chunks, embedding = model)
db

In [19]:
query = "It will be all the easier for us to conduct ourselves as belligerents"
retrieved_result = db.similarity_search(query)

In [20]:
print(retrieved_result)

[Document(metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'), Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemn